In [0]:
from pyspark.sql.functions import col, upper, date_format, to_date

###1. Reading data from ADLS Delta lake bronze layer

In [0]:
bronze_path = "abfss://bronze@migrationecom.dfs.core.windows.net/raw_orders"

df_bronze = spark.read.format("delta")\
                    .load(bronze_path)

In [0]:
df_bronze.display()

###2. DEDUPLICATION

In [0]:
initial_count = df_bronze.count()

df_deduped = df_bronze.dropDuplicates(['order_id'])

final_count = df_deduped.count()

if initial_count != final_count:
    print(f"[INFO] Removed {initial_count} - {final_count} duplicate records.")

###3. FORMATTING & DATA QUALITY

#### Formatting

In [0]:
df_formatted = df_deduped.withColumnRenamed("status", "order_status")\
                        .withColumn("order_status", upper(col("order_status")))\
                        .withColumn("order_date", date_format(to_date(col("order_date"), "MM-dd-yyyy"), "dd-MM-yyyy"))

In [0]:
df_formatted.display()

#### Data Quality

In [0]:
valid_statuses = ["DELIVERED", "SHIPPED", "PENDING", "CANCELLED", "PROCESSING", "CARTABANDONED"]
dq_conditions = col("order_id").isNotNull() & col("order_status").isin(valid_statuses)

df_good_data = df_formatted.filter(dq_conditions)
df_bad_data = df_formatted.filter(~dq_conditions)

In [0]:
df_good_data.display()

In [0]:
df_bad_data.display()

### 4. Route Data to Respective ADLS Containers

In [0]:
if df_bad_data.count() > 0:
    quarantine_path = "abfss://quarantine@migrationecom.dfs.core.windows.net/bad_orders_data"
    df_bad_data.write.format("delta")\
        .mode("append")\
        .save(quarantine_path)
    print(f"[WARNING] Invalid records quarantined in ADLS {quarantine_path}")

silver_path = "abfss://silver@migrationecom.dfs.core.windows.net/cleansed_orders"
df_good_data.write.format("delta")\
            .mode("overwrite")\
            .save(silver_path)
print(f"[SYSTEM] Silver processing complete. Clean data saved in ADLS: {silver_path}")